In [1]:
from pathlib import Path
import json
import csv

In [3]:
query_dir = Path("/content/drive/MyDrive/프젝랩/Models/query")

query_files = sorted(query_dir.glob("document_query_chunk_*_with_queries.json"))

all_items = []
all_queries = []

for query_file in query_files:
  with open(query_file, "r", encoding="utf-8") as f:
    data = json.load(f)

  items = data.get("items")

  if not isinstance(items, list):
    continue

  for item in items:
    record_id = item.get("record_id")
    concat = item.get("concat")
    queries = item.get("query")

    all_items.append({
        "record_id": record_id,
        "concat": concat,
        "query": queries
    })

    if not isinstance(queries, list) or len(queries) < 3:
      continue

    for idx in [0, 1, 2]:
      query_text = queries[idx]

      all_queries.append({
          "record_id": record_id,
          "query_idx": idx,
          "query": str(query_text).strip(),
          "concat": concat
      })

print(len(query_files))
print(len(all_items))
print(len(all_queries))
print(all_queries[:30])


merged_items_path = Path("/content/drive/MyDrive/프젝랩/Models/merged_items.json")

with open(merged_items_path, "w", encoding="utf-8") as f:
  json.dump(all_items, f, ensure_ascii=False, indent=2)

merged_queries_path = Path("/content/drive/MyDrive/프젝랩/Models/merged_queries.csv")

with open(merged_queries_path, "w", encoding="utf-8-sig", newline="") as f:
  writer =  csv.DictWriter(
      f,
      fieldnames=[
          "record_id",
          "query_idx",
          "query",
          "concat"
      ]
  )
  writer.writeheader()
  writer.writerows(all_queries)

KeyboardInterrupt: 

generated query 기반 multi-representation retrieval

각 콘텐츠로부터 유형별로 생성된 3가지 query에 대해 원 query와 유사도가 min_score 이상인 경우에만 해당 유형으로 추가적으로 routing

원 query를 포함한 총 4개 유형의 query에 대해 query level rrf 수행

original_query: expaned_query = 0.4:0.6

0.6은 3가지 유형의 query들이 나눠가짐 (min_score 충족 못한 query유형 존재가능하므로)

In [19]:
min_scores = [0.50, 0.60, 0.65, 0.70]

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

merged_queries_path = Path("/content/drive/MyDrive/프젝랩/Models/merged_queries.csv")

query_bank_df = pd.read_csv(merged_queries_path)
query_bank_df = query_bank_df.dropna(subset=['record_id', 'query_idx', 'query']).copy()
query_bank_df['query_idx'] = query_bank_df['query_idx'].astype(int)
query_bank_df['query'] = query_bank_df['query'].astype(str).str.strip()

print(query_bank_df.shape)
print(query_bank_df['query_idx'].value_counts().sort_index())
query_bank_df.head()



(14490, 4)
query_idx
0    4830
1    4830
2    4830
Name: count, dtype: int64


,record_id,query_idx,query,concat
0,list1_52380,0,부정수급 사례 예방,[TITLE] 부정수급 위험 사이렌 [2026-04호]\n[SUMMARY] 재정지원...
1,list1_52380,1,재정지원사업에서 부정수급을 막는 사례 자료를 찾고 싶어요,[TITLE] 부정수급 위험 사이렌 [2026-04호]\n[SUMMARY] 재정지원...
2,list1_52380,2,부정수급 위험 사이렌 재정지원사업 부정수급 예방,[TITLE] 부정수급 위험 사이렌 [2026-04호]\n[SUMMARY] 재정지원...
3,list1_52330,0,철골작업 사망사고,[TITLE] 건설업 사망사고 유발 SIF(철골작업)\n[SUMMARY] 건설현장 ...
4,list1_52330,1,건설현장에서 철골 작업 중 큰 사고가 나는 위험요인을 알고 싶어요,[TITLE] 건설업 사망사고 유발 SIF(철골작업)\n[SUMMARY] 건설현장 ...


In [5]:
!pip -q install rank_bm25 konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 31.0 MB/s eta 0:00:00


In [6]:
import json
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from konlpy.tag import Komoran

try:
  import chromadb
except ImportError:
  !pip install chromadb
  import chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 805.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Fou

In [7]:
model_name = 'koe5'
hf_id = 'nlpai-lab/KoE5'
db_path = '/content/drive/MyDrive/프젝랩/임베딩 모델/chroma_db/koe5'

top_k = 6

In [8]:
client = chromadb.PersistentClient(path=db_path)
collection = client.get_collection("client")

In [9]:
all_data = collection.get()
ids = all_data['ids']
docs = all_data['documents']
metadatas = all_data['metadatas']


In [10]:
id_to_doc = {doc_id: doc for doc_id, doc in zip(ids, docs)}
id_to_meta = {doc_id: meta for doc_id, meta in zip(ids, metadatas)}

In [11]:
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('huggingface')
login(token=HF_TOKEN)

In [12]:
embedding_model = SentenceTransformer(hf_id, device='cpu', token=HF_TOKEN)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [13]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_model = embedding_model.to(device)

query_bank_save_dir = Path("/content/drive/MyDrive/프젝랩/Models")
query_bank_save_dir.mkdir(parents=True, exist_ok=True)

query_bank_meta_path = query_bank_save_dir / "query_metadata.csv"
query_bank_emb_path = query_bank_save_dir / "query_embeddings.npy"

if query_bank_meta_path.exists() and query_bank_emb_path.exists():
  query_bank_df = pd.read_csv(query_bank_meta_path)
  query_embeddings = np.load(query_bank_emb_path)

else:
  query_texts = query_bank_df['query'].tolist()

  query_embeddings = embedding_model.encode(query_texts, batch_size=128 if device =='cuda' else 32, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)
  query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

  query_bank_df.to_csv(query_bank_meta_path, index=False, encoding="utf-8-sig")
  np.save(query_bank_emb_path, query_embeddings)

print(query_embeddings.shape)

(14490, 1024)


In [14]:
query_types_indices = {}

for idx in [0, 1, 2]:
  indices = query_bank_df.index[query_bank_df['query_idx'] == idx].to_numpy()
  query_types_indices[idx] = indices
  print(f'type {idx} 개수: {len(indices)}')

type 0 개수: 4830
type 1 개수: 4830
type 2 개수: 4830


In [15]:
def route_query(user_query, min_score, top_k_per_type=5, return_topk=False):
  user_query_emb = embedding_model.encode([user_query], normalize_embeddings=True)[0]

  routed = []
  debug_rows = []

  for idx in [0, 1, 2]:
    query_idxs = query_types_indices[idx]
    type_embs = query_embeddings[query_idxs]

    sim_score = type_embs @ user_query_emb
    order = np.argsort(sim_score)[::-1][:top_k_per_type]
    type_candidates = []

    for rank, local_idx in enumerate(order, 1):
      global_idx = query_idxs[local_idx]
      score = float(sim_score[local_idx])
      row = query_bank_df.loc[global_idx]

      candidate = {
          "query_idx": idx,
          "rank_in_type": rank,
          "similarity": score,
          "record_id": row['record_id'],
          "routed_query": row['query'],
          'content': row['concat']
      }

      debug_rows.append(candidate)

      if score >= min_score:
        type_candidates.append(candidate)

    if return_topk:
      routed.extend(type_candidates)
    else:
      if len(type_candidates) > 0:
        routed.append(type_candidates[0])

  return routed, pd.DataFrame(debug_rows)

In [17]:
routed, debug_df = route_query(user_query="밀폐공간 질식사고 예방자료 검색", min_score=0.60, top_k_per_type=5, return_topk=False)

routed

[{'query_idx': 0,
  'rank_in_type': 1,
  'similarity': 0.7060918211936951,
  'record_id': 'list3_38347',
  'routed_query': '밀폐공간 질식 예방',
  'content': '[TITLE] 밀폐공간작업 특성별 질식재해예방 매뉴얼(TOP7)\n[SUMMARY] 산업안전보건기준에 관한 규칙 별표18(밀폐공간) 18호에 해당하는 밀폐공간작업시 질식재해를 예방하기 위한 매뉴얼'},
 {'query_idx': 1,
  'rank_in_type': 1,
  'similarity': 0.8581956624984741,
  'record_id': 'list1_50064',
  'routed_query': '밀폐공간에서 질식 사고가 나지 않게 하는 예방자료를 찾고 있어요',
  'content': '[TITLE] 키메세지(8호_질식)\n[SUMMARY] [계절·시기별] 질식 사고 예방대책 및 관련 콘텐츠 안내 등'},
 {'query_idx': 2,
  'rank_in_type': 1,
  'similarity': 0.7813876867294312,
  'record_id': 'list3_43647',
  'routed_query': '밀폐공간 질식재해예방 안전작업 가이드',
  'content': '[TITLE] 밀폐공간 질식재해예방 안전작업 가이드\n[SUMMARY] 밀폐공간 질식재해예방 안전작업 가이드 개정'}]

In [16]:
def threshold_diagnostic(test_dataset, thresholds=(0.60, 0.65, 0.70, 0.75)):
    rows = []

    for min_score in thresholds:
        selected_counts = []
        selected_by_type = {0: 0, 1: 0, 2: 0}

        for item in tqdm(test_dataset, desc=f"threshold={min_score}"):
            query = item["query"]

            routed, _ = route_query(
                query,
                min_score=min_score,
                top_k_per_type=5,
                return_topk=False
            )

            selected_counts.append(len(routed))

            for r in routed:
                selected_by_type[r["query_idx"]] += 1

        rows.append({
            "min_score": min_score,
            "test_size": len(test_dataset),
            "avg_selected_queries": round(float(np.mean(selected_counts)), 3),
            "no_routing_count": int(sum(c == 0 for c in selected_counts)),
            "no_routing_ratio": round(sum(c == 0 for c in selected_counts) / len(selected_counts) * 100, 2),
            "select_1_count": int(sum(c == 1 for c in selected_counts)),
            "select_2_count": int(sum(c == 2 for c in selected_counts)),
            "select_3_count": int(sum(c == 3 for c in selected_counts)),
            "query0_selected_count": selected_by_type[0],
            "query1_selected_count": selected_by_type[1],
            "query2_selected_count": selected_by_type[2],
        })

    return pd.DataFrame(rows)

In [17]:
test_dataset_path = Path("/content/drive/MyDrive/프젝랩/test_dataset_수정.json")

with open(test_dataset_path, 'r', encoding="utf-8") as f:
  test_dataset = json.load(f)

In [20]:
threshold_df = threshold_diagnostic(test_dataset, thresholds=min_scores)
threshold_df

threshold=0.5:   0%|          | 0/28 [00:00<?, ?it/s]

threshold=0.6:   0%|          | 0/28 [00:00<?, ?it/s]

threshold=0.65:   0%|          | 0/28 [00:00<?, ?it/s]

threshold=0.7:   0%|          | 0/28 [00:00<?, ?it/s]

,min_score,test_size,avg_selected_queries,no_routing_count,no_routing_ratio,select_1_count,select_2_count,select_3_count,query0_selected_count,query1_selected_count,query2_selected_count
0,0.50,28,3.000,0,0.00,0,0,28,28,28,28
1,0.60,28,2.750,0,0.00,2,3,23,26,25,26
2,0.65,28,2.214,3,10.71,3,7,15,23,21,18
3,0.70,28,1.679,5,17.86,8,6,9,16,17,14


In [21]:
komoran = Komoran()

KEEP_POS = {"NNG", "NNP", "SL", "XR"}
# NNG: 일반명사, NNP: 고유명서, SL: 외국어/영어 약어, XR: 어근

EXCLUDE_TOKENS = {"TITLE", "SUMMARY", "예방", "자료", "가이드", "안전", "가이드라인", "보건"}

def tokenize(text):
  if text is None:
    return []

  tokens = []

  for word, pos in komoran.pos(str(text)):
    if pos in KEEP_POS:
      token = word.strip()

      if token in EXCLUDE_TOKENS:
        continue

      if token:
        tokens.append(token)

  return tokens

In [22]:
tokenized = []

for doc in tqdm(docs, desc="BM25 tokenizing"):
  tokenized.append(tokenize(doc))

print("sample tokens: ", tokenized[0][:30])

BM25 tokenizing:   0%|          | 0/8980 [00:00<?, ?it/s]

sample tokens:  ['부정', '수급', '위험', '사이렌', '정지원', '사업', '부정', '수급', '사례', '전파', '부정', '수급']


In [23]:
bm25 = BM25Okapi(tokenized)

In [24]:
def bm25_retriever(query, bm25_top_k, min_score=0):
  query_tokens = tokenize(query)
  scores = bm25.get_scores(query_tokens)
  top_indices = np.argsort(scores)[::-1][:bm25_top_k]

  bm25_ids = [ids[i] for i in top_indices if scores[i] > min_score]
  bm25_scores = [scores[i] for i in top_indices if scores[i] > min_score]

  return bm25_ids, bm25_scores

In [25]:
def dense_retriever(query, dense_top_k):
  query_embedding = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0].tolist()

  results = collection.query(
      query_embeddings=[query_embedding],
      n_results=dense_top_k,
      include=["documents", "metadatas", "distances"]
  )

  dense_ids = results["ids"][0]
  dense_distances = results["distances"][0]

  return dense_ids, dense_distances

In [26]:
def rrf_fusion(rankings, rrf_k, weights=None): # rankings: [[dense_id1, dense_id2, ...], [bm25_id1, bm25_id2, ...]]
  scores = {}

  if weights is None:
    weights = [1.0] * len(rankings)

  for ranking, weight in zip(rankings, weights):
    for rank, doc_id in enumerate(ranking, 1):
      if doc_id not in scores:
        scores[doc_id] = 0
      scores[doc_id] += weight * (1 / (rrf_k + rank))

  fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)

  return fused


In [27]:
def hybrid_retriever(query, dense_top_k=50, bm25_top_k=50, rrf_k1=10, weights1=[0.8, 0.2], candidate_k=10):
  dense_ids, dense_distances = dense_retriever(query, dense_top_k)
  bm25_ids, bm25_scores = bm25_retriever(query, bm25_top_k)

  fused = rrf_fusion([dense_ids, bm25_ids], rrf_k1, weights1)

  fused_ids = [doc_id for doc_id, score in fused][:candidate_k]

  return fused_ids

In [28]:
def multi_query_retriever(user_query, min_score=0.70, dense_top_k=50, bm25_top_k=50, rrf_k1=10, weights=[0.8, 0.2], candidate_k=10, rrf_k2=10, original_weight=0.4, expanded_weight=0.6, final_k=6):
  routed, route_debug_df = route_query(user_query, min_score, top_k_per_type=5, return_topk=False)

  queries = [user_query]
  query_sources = ["original"]

  for r in routed:
    queries.append(r['routed_query'])
    query_sources.append(f"query[{r['query_idx']}]")

  rankings = []

  for query in queries:
    ids_rank = hybrid_retriever(query, dense_top_k, bm25_top_k, rrf_k1, weights, candidate_k)

    rankings.append(ids_rank)

  if len(rankings) == 1:
    weights = [1.0]

  else:
    expanded_num = len(rankings) - 1
    each_expanded_weight = (1 - original_weight) / expanded_num
    weights = [original_weight] + [each_expanded_weight] * expanded_num

  fused = rrf_fusion(rankings, rrf_k2, weights)
  fused_ids = [doc_id for doc_id, score in fused[:final_k]]

  outputs = []
  for rank, doc_id in enumerate(fused_ids, 1):
    meta = id_to_meta[doc_id]
    doc = id_to_doc[doc_id]

    outputs.append({
        'rank': rank,
        'doc_id': doc_id,
        'title': meta.get('title'),
        'document': doc,
        'metadata': meta
    })

  debug_info = {
      'query_text': queries,
      'query_sources': query_sources,
      "query_weights": weights,
      "routed": routed,
      "route_debug_df": route_debug_df
  }

  return outputs, debug_info

In [29]:
user_query = "밀폐공간 질식사고 예방자료 검색"

min_score=0.70
dense_top_k=50
bm25_top_k=50
rrf_k1=10
weights=[0.8, 0.2]
candidate_k=10
final_k=6

rrf_k2 = 10
original_weight = 0.4
expanded_weight = 0.6

results, debug_info = multi_query_retriever(user_query=user_query,  min_score=min_score, dense_top_k=dense_top_k, bm25_top_k=bm25_top_k, rrf_k1=rrf_k1, weights=weights, candidate_k=candidate_k, rrf_k2=rrf_k2, original_weight=original_weight, expanded_weight=expanded_weight, final_k=final_k)

pd.DataFrame(results)[['rank', 'title', 'doc_id']]

,rank,title,doc_id
0,1,`24년 밀폐공간 질식재해예방 안전작업 가이드,list3_46430
1,2,밀폐공간 질식재해예방 안전작업 가이드,list3_43647
2,3,키메세지(8월_질식재해예방),list1_46399
3,4,[이지애의 안전있수다] 밀폐공간 질식재해예방(16개국어),list2_46245
4,5,[이지애의 안전있수다] 밀폐공간 질식재해예방,list2_45915
5,6,밀폐공간작업 특성별 질식재해예방 매뉴얼(TOP5),list3_38345


In [39]:
debug_info['query_text']

['밀폐공간 질식사고 예방자료 검색',
 '밀폐공간 질식 예방',
 '밀폐공간에서 질식 사고가 나지 않게 하는 예방자료를 찾고 있어요',
 '밀폐공간 질식재해예방 안전작업 가이드']

In [30]:
def evaluation(save_dir, test_dataset, min_score=0.70, dense_top_k=50, bm25_top_k=50, rrf_k1=10, weights=[0.8, 0.2], candidate_k=10, rrf_k2=10, original_weight=0.4, expanded_weight=0.6, final_k=6):
  rows = []

  for idx, test_data in enumerate(test_dataset, 1):
    user_query = test_data['query']
    gold_title = test_data['title']

    results, debug_info = multi_query_retriever(user_query, min_score, dense_top_k, bm25_top_k, rrf_k1, weights, candidate_k, rrf_k2, original_weight, expanded_weight, final_k)

    correct_rank = None

    for rank, result_item in enumerate(results, 1):
      title = result_item['title']
      if title == gold_title:
        correct_rank = rank
        break

    top_titles = []
    top_doc_ids = []

    for result_item in results:
      doc_id = result_item.get('doc_id')
      title = result_item.get('title')

      top_doc_ids.append(str(doc_id))
      top_titles.append(str(title))

    query_texts = debug_info.get('query_text')
    query_sources = debug_info.get('query_sources')
    query_weights = debug_info.get('query_weights')

    routed_queries = query_texts[1:] if len(query_texts) > 1 else []
    routed_sources = query_sources[1:] if len(query_texts) > 1 else []

    rows.append({
        "query_idx": idx,
        "query": user_query,
        "gold_title": gold_title,
        "gold_rank": correct_rank,
        "top1_hit": int(correct_rank == 1),
        "top6_hit": int(correct_rank is not None and correct_rank <= final_k),
        "selected_queries_num": len(routed_queries),
        "query_sources": ",".join(query_sources),
        "query_weights": str(query_weights),
        "routed_queries": "|".join(routed_queries),
        "routed_sources": ",".join(routed_sources),
        "top_titles": "|".join(top_titles),
        "top_doc_ids": " | ".join(top_doc_ids),
    })


  detail_df = pd.DataFrame(rows)
  n = len(detail_df)

  test_size = len(test_dataset)

  summary = {
      'setting': 'multi_query_before_rerank',
      'min_score': min_score,
      'test_size': test_size,
      'dense_top_k': dense_top_k,
      'bm25_top_k': bm25_top_k,
      'rrf_k1': rrf_k1,
      "weights1": str(weights),
      "candidate_k": candidate_k,
      "rrf_k2": rrf_k2,
      "original_weight": original_weight,
      "expanded_weight": expanded_weight,
      'final_k': final_k,

      "top1_hit": int(detail_df["top1_hit"].sum()),
      "top6_hit": int(detail_df["top6_hit"].sum()),
      "top1_accuracy": round(detail_df["top1_hit"].mean() * 100, 2),
      "top6_recall": round(detail_df["top6_hit"].mean() * 100, 2),

      "avg_selected_query_num": round(detail_df["selected_queries_num"].mean(), 3),
      "no_routing_count": int((detail_df["selected_queries_num"] == 0).sum()),
      "no_routing_ratio": round((detail_df["selected_queries_num"] == 0).mean() * 100, 2),
  }

  summary_df = pd.DataFrame([summary])

  if save_dir is not None:
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    detail_path = save_dir / f"multi_query_before_rerank_detail_min{min_score}_rrf_k2_{rrf_k2}_original_weight{original_weight}.csv"
    summary_path = save_dir / f"multi_query_before_rerank_summary_min{min_score}_rrf_k2_{rrf_k2}_original_weight{original_weight}.csv"

    detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print(f"detail saved: {detail_path}")
    print(f"summary saved: {summary_path}")

  return detail_df, summary


In [31]:
save_dir = "/content/drive/MyDrive/프젝랩/Eval results/multi_queries_before_rerank"
min_score=0.70
dense_top_k=50
bm25_top_k=50
rrf_k1=10
weights=[0.8, 0.2]
candidate_k=10
final_k=6

rrf_k2 = 10
original_weight = 0.4
expanded_weight = 0.6

detail_df, summary = evaluation(save_dir=save_dir, test_dataset=test_dataset, min_score=min_score, dense_top_k=dense_top_k, bm25_top_k=bm25_top_k, rrf_k1=rrf_k1, weights=weights, candidate_k=candidate_k, rrf_k2=rrf_k2, original_weight=original_weight, expanded_weight=expanded_weight, final_k=final_k)

summary

detail saved: /content/drive/MyDrive/프젝랩/Eval results/multi_queries_before_rerank/multi_query_before_rerank_detail_min0.7_rrf_k2_10_original_weight0.4.csv
summary saved: /content/drive/MyDrive/프젝랩/Eval results/multi_queries_before_rerank/multi_query_before_rerank_summary_min0.7_rrf_k2_10_original_weight0.4.csv


{'setting': 'multi_query_before_rerank',
 'min_score': 0.7,
 'test_size': 28,
 'dense_top_k': 50,
 'bm25_top_k': 50,
 'rrf_k1': 10,
 'weights1': '[0.8, 0.2]',
 'candidate_k': 10,
 'rrf_k2': 10,
 'original_weight': 0.4,
 'expanded_weight': 0.6,
 'final_k': 6,
 'top1_hit': 4,
 'top6_hit': 10,
 'top1_accuracy': np.float64(14.29),
 'top6_recall': np.float64(35.71),
 'avg_selected_query_num': np.float64(1.679),
 'no_routing_count': 5,
 'no_routing_ratio': np.float64(17.86)}

In [32]:
def run_multi_query_before_rerank_grid(
    test_dataset,
    min_scores=(0.65, 0.70),
    rrf_k2_values=(5, 10, 20, 30),
    weight_settings=((0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.7, 0.3)),
    dense_top_k=50,
    bm25_top_k=50,
    rrf_k1=10,
    weights1=[0.8, 0.2],
    candidate_k=10,
    final_k=6,
    save_dir=None
):
    rows = []

    for min_score in min_scores:
        for rrf_k2 in rrf_k2_values:
            for original_weight, expanded_weight in weight_settings:
                detail_df, summary = evaluation(
                    test_dataset=test_dataset,
                    min_score=min_score,
                    dense_top_k=dense_top_k,
                    bm25_top_k=bm25_top_k,
                    rrf_k1=rrf_k1,
                    weights=weights1,
                    candidate_k=candidate_k,
                    rrf_k2=rrf_k2,
                    original_weight=original_weight,
                    expanded_weight=expanded_weight,
                    final_k=final_k,
                    save_dir=None
                )

                rows.append(summary)

    grid_df = pd.DataFrame(rows)

    grid_df = grid_df.sort_values(
        by=["top6_recall", "top1_accuracy"],
        ascending=False
    )

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        grid_df.to_csv(
            save_dir / "multi_query_before_rerank_grid.csv",
            index=False,
            encoding="utf-8-sig"
        )

    return grid_df

In [33]:
grid_df = run_multi_query_before_rerank_grid(
    test_dataset=test_dataset,
    min_scores=(0.65, 0.70),
    rrf_k2_values=(5, 10, 20, 30),
    weight_settings=[
        (0.3, 0.7),
        (0.4, 0.6),
        (0.5, 0.5),
        (0.6, 0.4),
        (0.7, 0.3),
    ],
    save_dir="/content/drive/MyDrive/프젝랩/Eval results/multi_queries_before_rerank"
)

grid_df[[
    "min_score",
    "rrf_k2",
    "original_weight",
    "expanded_weight",
    "top1_accuracy",
    "top6_recall",
    "top1_hit",
    "top6_hit",
    "avg_selected_query_num",
    "no_routing_count"
]]

,min_score,rrf_k2,original_weight,expanded_weight,top1_accuracy,top6_recall,top1_hit,top6_hit,avg_selected_query_num,no_routing_count
0,0.65,5,0.3,0.7,17.86,35.71,5,10,2.214,3
5,0.65,10,0.3,0.7,17.86,35.71,5,10,2.214,3
6,0.65,10,0.4,0.6,17.86,35.71,5,10,2.214,3
10,0.65,20,0.3,0.7,17.86,35.71,5,10,2.214,3
11,0.65,20,0.4,0.6,17.86,35.71,5,10,2.214,3
12,0.65,20,0.5,0.5,17.86,35.71,5,10,2.214,3
13,0.65,20,0.6,0.4,17.86,35.71,5,10,2.214,3
16,0.65,30,0.4,0.6,17.86,35.71,5,10,2.214,3
17,0.65,30,0.5,0.5,17.86,35.71,5,10,2.214,3
18,0.65,30,0.6,0.4,17.86,35.71,5,10,2.214,3


multi query를 사용함으로써 top1_accuracy가 유의적으로 상승함

즉, 후보군에 있던 gold answer의 순위를 위로 올리는 데 도움됨

min_score = 0.65\
rrf2_k = 10 or 20

original_weight = 0.4\
expanded_weight = 0.6

In [34]:
save_dir = "/content/drive/MyDrive/프젝랩/Eval results/multi_queries_before_rerank"
min_score=0.65
dense_top_k=50
bm25_top_k=50
rrf_k1=10
weights=[0.8, 0.2]
candidate_k=10
final_k=6

rrf_k2 = 10
original_weight = 0.4
expanded_weight = 0.6

detail_df, summary = evaluation(save_dir=save_dir, test_dataset=test_dataset, min_score=min_score, dense_top_k=dense_top_k, bm25_top_k=bm25_top_k, rrf_k1=rrf_k1, weights=weights, candidate_k=candidate_k, rrf_k2=rrf_k2, original_weight=original_weight, expanded_weight=expanded_weight, final_k=final_k)

summary

detail saved: /content/drive/MyDrive/프젝랩/Eval results/multi_queries_before_rerank/multi_query_before_rerank_detail_min0.65_rrf_k2_10_original_weight0.4.csv
summary saved: /content/drive/MyDrive/프젝랩/Eval results/multi_queries_before_rerank/multi_query_before_rerank_summary_min0.65_rrf_k2_10_original_weight0.4.csv


{'setting': 'multi_query_before_rerank',
 'min_score': 0.65,
 'test_size': 28,
 'dense_top_k': 50,
 'bm25_top_k': 50,
 'rrf_k1': 10,
 'weights1': '[0.8, 0.2]',
 'candidate_k': 10,
 'rrf_k2': 10,
 'original_weight': 0.4,
 'expanded_weight': 0.6,
 'final_k': 6,
 'top1_hit': 5,
 'top6_hit': 10,
 'top1_accuracy': np.float64(17.86),
 'top6_recall': np.float64(35.71),
 'avg_selected_query_num': np.float64(2.214),
 'no_routing_count': 3,
 'no_routing_ratio': np.float64(10.71)}